In [1]:
# %load_ext autoreload
# %autoreload 2

In [2]:
from src.data import pointcloud_processing, timeseries_processing
from src.data import config, pc_statistics
from src.data.config import TARGET_EXTENTS_VIF, TARGET_EXTENTS_VIF_SPLITS

import pandas as pd
from pointcloudset import PointCloud

import pickle
import warnings

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import panel as pn

pn.extension("plotly")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


#### Load Met Data from Bagfile

In [246]:
BAG_NAME_REFERENCE = "molisens_met_2023_08_07-15_36_45_converted"
DATA_DIR_REFERENCE = config.INTERIM_DATA_FOLDER / "ViF_Roof" / "data"
df_reference = timeseries_processing.load(
    DATA_DIR_REFERENCE / BAG_NAME_REFERENCE,
    "/sensing/aws/ws100_measurements",
    config.PATH_TO_LUFFT_MSGS,
    timestamp_source="header",
)

BAG_NAME_RAIN = "molisens_met_2023_08_29-06_04_46_converted"
DATA_DIR_RAIN = config.INTERIM_DATA_FOLDER / "ViF_Roof" / "data"
df_rain = timeseries_processing.load(
    DATA_DIR_RAIN / BAG_NAME_RAIN,
    "/sensing/aws/ws100_measurements",
    config.PATH_TO_LUFFT_MSGS,
    timestamp_source="header",
)

# Shift to account for AWS averaging duration
df_rain.index.asfreq = "S"
df_rain.index = df_rain.index.round("S")
df_rain.loc[:, ("precipitation", "intensity_hour_shifted")] = df_rain.precipitation.intensity_hour.shift(
    periods=-60, freq="S"
)

Find the most relevant paramters for further analysis:

In [251]:
df_rain.precipitation.loc[:, ["absolute", "differential", "total_precipitation_particles", "total_drops"]].corrwith(
    df_rain.precipitation.intensity_hour_shifted
)

Parameter
absolute                         0.173719
differential                     0.513961
total_precipitation_particles    0.267367
total_drops                      0.355501
dtype: float64

In [258]:
fig1 = px.line(df_rain.precipitation.absolute)
fig1.update_traces(line=dict(color="blue"), connectgaps=True)
fig2 = px.line(df_rain.precipitation.differential.resample("10s").sum())
fig2.update_traces(line=dict(color="red"))
fig3 = px.line(df_rain.precipitation.intensity_hour_shifted)
fig3.update_traces(line=dict(color="green"), connectgaps=True)
# fig6 = px.line(df_rain.precipitation.intensity_hour)
# fig6.update_traces(line=dict(color="lightgreen"))
fig4 = px.line(df_rain.precipitation.total_precipitation_particles.resample("10s").sum())
fig4.update_traces(line=dict(color="gold"))
fig5 = px.line(df_rain.precipitation.total_drops.resample("10s").sum())
fig5.update_traces(line=dict(color="black"))

# Create a subplot with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add the traces from fig1, fig3, fig4, and fig5 to the primary y-axis
for trace in fig1.data + fig3.data + fig4.data + fig5.data:
    fig.add_trace(trace, secondary_y=False)

# Add the traces from fig2 to the secondary y-axis
for trace in fig2.data:
    fig.add_trace(trace, secondary_y=True)

# Update the layout
fig.update_layout(width=1400, height=600)

fig.show()

In [260]:
# w_differ = df_rain.precipitation.differential.resample("10s").sum()
# w_int_h = df_rain.precipitation.intensity_hour.resample("1S").mean()
# # Shift to account for the AWS averaging duration
# w_int_h = w_int_h.shift(periods=-60, freq="S")

RESAMPLE_FREQ = "10s"
columns_and_aggregation = {
    "intensity_hour_shifted": "mean",
    "absolute": "sum",
    "differential": "sum",
    "total_precipitation_particles": "sum",
    "total_drops": "sum",
}

df_rain_relevant = df_rain.precipitation.loc[:, columns_and_aggregation.keys()]
df_rain_relevant = df_rain_relevant.resample(RESAMPLE_FREQ).agg(columns_and_aggregation)
df_rain_relevant.corr()

Parameter,intensity_hour_shifted,absolute,differential,total_precipitation_particles,total_drops
Parameter,,,,,
intensity_hour_shifted,1.000000,0.176823,0.928506,0.500550,0.648266
absolute,0.176823,1.000000,0.148056,0.224661,0.185971
differential,0.928506,0.148056,1.000000,0.523736,0.697280
total_precipitation_particles,0.500550,0.224661,0.523736,1.000000,0.835392
total_drops,0.648266,0.185971,0.697280,0.835392,1.000000


### Load PC data

In [6]:
TOPICS = {
    "lid_pts": "/sensing/lidar/points",
    "lid_pts2": "/sensing/lidar/points2",
    "rad_pts": "/sensing/radar/points",
}

In [7]:
dataset_reference = pointcloud_processing.load_pointcloudset(
    DATA_DIR_REFERENCE, BAG_NAME_REFERENCE, topic=TOPICS["lid_pts"], verbose=True, invert_axes=["x", "y"]
)
dataset_rain = pointcloud_processing.load_pointcloudset(
    DATA_DIR_RAIN, BAG_NAME_RAIN, topic=TOPICS["lid_pts"], verbose=True, invert_axes=["x", "y"]
)

Searching for pointcloudset files in:
/workspaces/MOLISENSext_analysis/data/2interim/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_0
8_07-15_36_45_converted

Dataset loaded from:
/workspaces/MOLISENSext_analysis/data/2interim/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_08_07-15_36_45_converted
start =    2023-08-07 13:36:48.072408
end =      2023-08-07 13:39:47.651742
duration = 0:02:59.579334
length =   1795
avg frequency =  10.00 Hz
Inverting axes: ['x', 'y']


Searching for pointcloudset files in:
/workspaces/MOLISENSext_analysis/data/2interim/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_0
8_29-06_04_46_converted

Dataset loaded from:
/workspaces/MOLISENSext_analysis/data/2interim/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_08_29-06_04_46_converted
start =    2023-08-29 04:05:19.853657
end =      2023-08-29 05:04:46.069975
duration = 0:59:26.216318
length =   35659
avg frequency =  10.00 Hz
Inverting axes: ['x', 'y']


### Resample Dataset to 1min

In [8]:
# Remove the first 20 seconds of the dataset. This should be done when loading data !!!!
path = config.PROCESSED_DATA_FOLDER / "rain_minutes.pickle"
if not path.exists():
    rain_ds_minutes = pointcloud_processing.resample_dataset(dataset_rain[20 * 10 :], "1min", statistics=["std", "sum"])

    # Save the resampled datasets to pickle
    with open(path, "wb") as handle:
        pickle.dump(rain_ds_minutes, handle, protocol=pickle.HIGHEST_PROTOCOL)
else:
    # load from pickle
    path = config.PROCESSED_DATA_FOLDER / "rain_minutes.pickle"
    with open(path, "rb") as handle:
        rain_ds_minutes = pickle.load(handle)

## Target Statistics

In [9]:
rain_ds_minutes_mean = {}
for target_name, target_limits in TARGET_EXTENTS_VIF.items():
    rain_ds_minutes_mean[target_name] = pd.concat(
        rain_ds_minutes["mean"].apply(target_limits.apply_limits).apply(pc_statistics.mean_intensity)
    )

### Result plots

In [261]:
colors = [
    "#1f77b4",  # muted blue
    "#ff7f0e",  # safety orange
    "#2ca02c",  # cooked asparagus green
    "#d62728",  # brick red
    "#9467bd",  # muted purple
    "#8c564b",  # chestnut brown
]

fig = make_subplots(specs=[[{"secondary_y": True}]])
# fig.add_trace(go.Scatter(x=w_differ.index, y=w_differ*200, mode='lines', name='differential precipitation', line=dict(color='red')))
fig.add_trace(
    go.Scattergl(
        x=df_rain_relevant.intensity_hour_shifted.index,
        y=df_rain_relevant.intensity_hour_shifted,
        mode="lines",
        name="Rainfall Rate",
        line=dict(color="#000080", width=3),
    )
)

for col, (target_name, rain_ds_minutes_mean_ds) in zip(colors, rain_ds_minutes_mean.items()):
    fig.add_trace(
        go.Scatter(
            x=rain_ds_minutes_mean_ds.index,
            y=rain_ds_minutes_mean_ds,
            mode="lines",
            name=target_name,
            line=dict(color=col, width=2),
        ),
        secondary_y=True,
    )

fig.update_layout(
    template="ggplot2",
    autosize=False,
    width=1000,
    height=500,
    # title="Rainfall Rate vs. Noise in Pointclouds",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.8),
    xaxis_title="",
    yaxis_title="Rainfall Rate [mm/h]",
    yaxis2_title="Mean Intensity",
    font=dict(size=16),
)
fig.update_yaxes(title_font_color="#000080")
fig.update_yaxes(title_font_color="#d5196e", secondary_y=True)

#### Plot only parts

In [262]:
colors = [
    "#1f77b4",  # muted blue
    "#ff7f0e",  # safety orange
    "#2ca02c",  # cooked asparagus green
    "#d62728",  # brick red
    "#9467bd",  # muted purple
    "#8c564b",  # chestnut brown
]

fig = make_subplots(specs=[[{"secondary_y": True}]])
# fig.add_trace(go.Scatter(x=w_differ.index, y=w_differ*200, mode='lines', name='differential precipitation', line=dict(color='red')))
fig.add_trace(
    go.Scattergl(
        x=df_rain_relevant.intensity_hour_shifted.index,
        y=df_rain_relevant.intensity_hour_shifted,
        mode="lines",
        name="Rainfall Rate",
        line=dict(color="#000080", width=3),
    )
)

for col, (target_name, rain_ds_minutes_mean_ds) in zip(colors, rain_ds_minutes_mean.items()):
    if target_name in ["Target 2", "Ring Sensor"]:
        continue
    fig.add_trace(
        go.Scatter(
            x=rain_ds_minutes_mean_ds.index,
            y=rain_ds_minutes_mean_ds,
            mode="lines",
            name=target_name,
            line=dict(color=col, width=2),
        ),
        secondary_y=True,
    )

fig.update_layout(
    template="ggplot2",
    autosize=False,
    width=1000,
    height=500,
    # title="Rainfall Rate vs. Noise in Pointclouds",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.8),
    xaxis_title="",
    yaxis_title="Rainfall Rate [mm/h]",
    yaxis2_title="Mean Intensity",
    font=dict(size=16),
)
fig.update_yaxes(title_font_color="#000080")
fig.update_yaxes(title_font_color="#d5196e", secondary_y=True)

### Spearate by intensity

In [264]:
def compute_target_intensity(dataset, splits, return_type="dict"):
    intensities_dict = {}
    for target_name, dic in splits.items():
        intensities_dict[target_name] = {}
        for col, target_limits in dic.items():
            with warnings.catch_warnings(record=True):
                intensities_dict[target_name][col] = pd.concat(
                    dataset.apply(target_limits.apply_limits).apply(pc_statistics.mean_intensity)
                )
    if return_type == "dict":
        return intensities_dict

    if return_type == "df":
        return pd.concat(
            [pd.DataFrame(v) for v in intensities_dict.values()],
            axis=1,
            keys=intensities_dict.keys(),
        )


rain_ds_minutes_mean_intens = compute_target_intensity(
    rain_ds_minutes["mean"], TARGET_EXTENTS_VIF_SPLITS, return_type="dict"
)

In [265]:
#! DEPRECATED: but might be useful in the future as it uses go.scatter


colors = [
    "#1f77b4",  # muted blue
    "#ff7f0e",  # safety orange
    "#2ca02c",  # cooked asparagus green
    "#d62728",  # brick red
    "#9467bd",  # muted purple
    "#8c564b",  # chestnut brown
]
dashes = {"white": None, "grey": "dash", "black": "dot"}

fig = make_subplots(specs=[[{"secondary_y": True}]])
# fig.add_trace(go.Scatter(x=w_differ.index, y=w_differ*200, mode='lines', name='differential precipitation', line=dict(color='red')))

fig.add_trace(
    go.Scattergl(
        x=df_rain_relevant.intensity_hour_shifted.index,
        y=df_rain_relevant.intensity_hour_shifted,
        mode="lines",
        name="Rainfall Rate",
        line=dict(color="#000080", width=3),
    )
)


for plot_col, (target_name, dic) in zip(colors, rain_ds_minutes_mean_intens.items()):
    for col, rain_ds_minutes_mean_ds in dic.items():
        fig.add_trace(
            go.Scatter(
                x=rain_ds_minutes_mean_ds.index,
                y=rain_ds_minutes_mean_ds,
                mode="lines",
                name=target_name + col,
                line=dict(color=plot_col, width=2, dash=dashes[col]),
            ),
            secondary_y=True,
        )

fig.update_layout(
    template="ggplot2",
    autosize=False,
    width=1000,
    height=700,
    # title="Rainfall Rate vs. Noise in Pointclouds",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.8),
    xaxis_title="",
    yaxis_title="Rainfall Rate [mm/h]",
    yaxis2_title="Mean Intensity",
    font=dict(size=16),
)
fig.update_yaxes(title_font_color="#000080")
fig.update_yaxes(title_font_color="#d5196e", secondary_y=True)

In [284]:
rain_ds_minutes_mean_intens = compute_target_intensity(
    rain_ds_minutes["mean"], TARGET_EXTENTS_VIF_SPLITS, return_type="df"
)

In [285]:
# normalize the data
def normalize_df(df, kind="standard"):
    df = df.copy()
    if kind == "standard":
        return (df - df.mean()) / df.std()
    if kind == "minmax":
        return (df - df.min()) / (df.max() - df.min())


rain_ds_minutes_mean_intens_norm = normalize_df(rain_ds_minutes_mean_intens, kind="standard")
df_rain_relevant_norm = normalize_df(df_rain_relevant, kind="standard")

In [317]:
colors = [
    "#1f77b4",  # muted blue
    "#ff7f0e",  # safety orange
    "#2ca02c",  # cooked asparagus green
    "#d62728",  # brick red
    "#9467bd",  # muted purple
    "#8c564b",  # chestnut brown
]
dashes = {"white": None, "grey": "dash", "black": "dot"}

fig = make_subplots(specs=[[{"secondary_y": True}]])
# fig.add_trace(go.Scatter(x=w_differ.index, y=w_differ*200, mode='lines', name='differential precipitation', line=dict(color='red')))
fig = px.line(
    rain_ds_minutes_mean_intens_norm.melt(
        var_name=["target", "color"], value_name="intensity", ignore_index=False
    ).reset_index(),
    x="index",
    y="intensity",
    color="target",
    line_dash="color",
    line_group="color",
)

fig.add_trace(
    go.Scattergl(
        x=df_rain_relevant_norm.intensity_hour_shifted.index,
        y=df_rain_relevant_norm.intensity_hour_shifted,
        mode="lines",
        name="Rainfall Rate",
        line=dict(color="#000080", width=3),
    )
)


fig.update_layout(
    template="ggplot2",
    autosize=False,
    width=1000,
    height=700,
    # title="Rainfall Rate vs. Noise in Pointclouds",
    # legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=0.8),
    xaxis_title="",
    yaxis_title="Rainfall Rate [mm/h]",
    # yaxis2_title="Mean Intensity",
    font=dict(size=16),
)
fig.update_yaxes(title_font_color="#000080")
fig.update_yaxes(title_font_color="#d5196e", secondary_y=True)


def compress_legend(fig):
    group1_base, group2_base = fig.data[0].name.split(",")
    lines_marker_name = []
    start_x = []
    for i, trace in enumerate(fig.data):
        start_x.append(trace.x.min())
        part1, part2 = trace.name.split(",")
        if part1 == group1_base:
            lines_marker_name.append(
                {
                    "line": trace.line.to_plotly_json(),
                    "marker": trace.marker.to_plotly_json(),
                    "mode": trace.mode,
                    "name": part2.lstrip(" "),
                }
            )
        if part2 != group2_base:
            trace["name"] = ""
            trace["showlegend"] = False
        else:
            trace["name"] = part1

    ## Add the line/markers for the 2nd group
    for lmn in lines_marker_name:
        lmn["line"]["color"] = "black"
        lmn["marker"]["color"] = "black"
        fig.add_trace(go.Scatter(x=[min(start_x)], y=[None], **lmn))
    fig.update_layout(legend_title_text="", legend_itemclick=False, legend_itemdoubleclick=False)
    return fig


# Todo Handle exceptions
# compress_legend(fig)
fig

In [274]:
correlation = rain_ds_minutes_mean_intens_norm.corrwith(df_rain_relevant_norm.intensity_hour_shifted)

# Print the correlation values
print(correlation)

Target 1  white   -0.554906
          grey    -0.458438
          black   -0.482708
Target 2  white   -0.658301
          grey    -0.561343
          black   -0.353355
Target 3  white   -0.660079
          grey    -0.639066
          black   -0.621131
Target 4  white   -0.660488
          grey    -0.522275
          black   -0.613583
Target 5  white   -0.156022
          grey    -0.206373
          black    0.012656
dtype: float64


In [277]:
rain_ds_minutes_mean_intens_norm = normalize_df(rain_ds_minutes_mean_intens, kind="standard")
w_int_h_norm = normalize_df(df_rain_relevant.intensity_hour_shifted, kind="standard")
rain_ds_minutes_mean_intens_norm.columns = [
    "_".join(col).strip() for col in rain_ds_minutes_mean_intens_norm.columns.values
]

rain_ds_minutes_mean_intens_norm = pd.concat(
    [rain_ds_minutes_mean_intens_norm, w_int_h_norm.resample("1min").mean()], axis=1
)

In [281]:
fig = px.scatter_matrix(
    rain_ds_minutes_mean_intens_norm,
    dimensions=[
        "Target 1_white",
        "Target 2_white",
        "Target 3_white",
        "Target 4_white",
        "Target 5_white",
        "intensity_hour_shifted",
    ],
    color="intensity_hour_shifted",
)
fig.update_traces(diagonal_visible=False)
fig.show()

/opt/conda/lib/python3.10/site-packages/plotly/express/_core.py:279: FutureWarning:

iteritems is deprecated and will be removed in a future version. Use .items instead.



In [ ]:
# TODO: Check for best correlations between intesity values and rain_df_relevant